# LiPAD — Corrosion Detection Training

Train a **YOLO segmentation** corrosion detector with Roboflow-aligned settings.

## Quick start (Google Colab)

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Upload or clone `LIPAD_YOLO_TRAINING` into `/content/LIPAD_YOLO_TRAINING`.
3. Put raw images/labels under `corrosion_detection/datasets_raw/` **or** directly in `corrosion_detection/datasets/images/{train,val}` with matching `labels/{train,val}`.
4. Run all cells top to bottom.
5. Best weights are saved under `corrosion_detection/runs/<model>/corrosion_<model>_seg/weights/best.pt`.

## Configuration applied

| Setting | Value |
|---|---|
| Epochs | 100 |
| Learning rate | 0.01 |
| Optimizer | SGD |
| Image size | 640×640 (stretch) |
| Auto-orient | Yes |
| Contrast | Adaptive equalization (CLAHE) |
| Classes | fair, poor, severe (2 remapped, 3 dropped) |
| Aug copies (train) | 3 per source image (offline photometric) |
| Flips | Horizontal + vertical |
| Rotation | ±15° |
| Exposure | ±25% |
| Blur | up to 2.5 px |
| Noise | up to 10% of pixels |

Edit defaults in `shared/corrosion_config.py` if your Roboflow export uses different class ids.

In [1]:
%pip install torch torchvision
%pip install ultralytics pyyaml albumentations opencv-python-headless pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.6 MB/s eta 0:00:00


In [ ]:
# @title 1. Environment setup
import sys
from pathlib import Path

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content
    # After cloning/uploading, point here if needed:
    # %cd /content/LIPAD_YOLO_TRAINING
else:
    ROOT = Path(r'C:/Users/Admin/PROJECT_LIPAD/Corrosion/LIPAD_YOLO_TRAINING')
    if (Path.cwd() / 'shared').exists():
        ROOT = Path.cwd()
    elif (Path.cwd().parent / 'shared').exists():
        ROOT = Path.cwd().parent
    %cd {ROOT}
    sys.path.insert(0, str(ROOT))
    print('Repo root:', ROOT)


Mounted at /content/drive
/content


In [26]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!ls -lah /content/drive/MyDrive

total 1.1G
-rw------- 1 root root  180 Jan 18  2021 'ACTIVITY 1  ATP-ADP CYCLE (1).docx.gdoc'
-rw------- 1 root root  180 Jan 18  2021 'ACTIVITY 1  ATP-ADP CYCLE.docx.gdoc'
-rw------- 1 root root  180 Feb  1  2021 'ACTIVITY 2 CHLOROPHYLL AND OTHER PIGMENTS.gdoc'
-rw------- 1 root root 1.6M Oct  9  2020 'Activity 2 Gen Bio.docx'
-rw------- 1 root root  180 Feb 10  2021 'ACTIVITY 3 LIGHT DEPENDENT REACTION (1).gdoc'
-rw------- 1 root root  180 Feb 10  2021 'ACTIVITY 4 CALVIN CYCLE (1).gdoc'
-rw------- 1 root root  180 Jan 29  2020 'Book report.gslides'
-rw------- 1 root root  180 Jun 25  2021 'CHAPTER IV.gdoc'
-rw------- 1 root root  180 Jun 25  2021 'CHAPTER V.gdoc'
-rw------- 1 root root  180 Feb  7  2021 'Chlorophyl and other pigments.gslides'
drwx------ 2 root root 4.0K May 16  2020  Classroom
-rw------- 1 root root 532K Oct 19  2020 'Concept map subsystem.docx'
-rw------- 1 root root  180 Jun 23  2021 'Contact Information (1).gform'
-rw------- 1 root root  180 Dec 18  2021 'Contact 

In [8]:
from pathlib import Path

DATASET = Path("/content/LIPAD_CORROSION_DATASET")

for split in ["train", "val", "test"]:
    images = DATASET / "images" / split
    labels = DATASET / "labels" / split

    image_count = len(list(images.glob("*"))) if images.exists() else 0
    label_count = len(list(labels.glob("*"))) if labels.exists() else 0

    print(f"{split}:")
    print(f"  images: {image_count}")
    print(f"  labels: {label_count}")

train:
  images: 5163
  labels: 5163
val:
  images: 223
  labels: 223
test:
  images: 115
  labels: 115


In [ ]:
DATASET = TARGET / "datasets/corrosion/dataset"

print("Dataset exists:", DATASET.exists())

for split in ["train", "val", "test"]:
    images = DATASET / "images" /
     split
    labels = DATASET / "labels" / split

    print(
        f"{split}: "
        f"{len(list(images.glob('*'))) if images.exists() else 0} images, "
        f"{len(list(labels.glob('*'))) if labels.exists() else 0} labels"
    )

Dataset exists: True
train: 5163 images, 5163 labels
val: 223 images, 223 labels
test: 115 images, 115 labels


In [ ]:
from pathlib import Path

LOCAL_DATASET = Path("/content/LIPAD_CORROSION_DATASET")
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

print("Local dataset:", LOCAL_DATASET)

Local dataset: /content/LIPAD_CORROSION_DATASET


In [5]:
!mkdir -p /content/LIPAD_CORROSION_DATASET

!rsync -a --info=progress2 \
"/content/drive/.shortcut-targets-by-id/1EJ8w_u_xG5uLul0VbLJp5J8CwBzCGbW5/LIPAD_TRAINING_VERSION2/datasets/corrosion/dataset/" \
"/content/LIPAD_CORROSION_DATASET/"

    831,972,240 100%  180.67kB/s    1:14:56 (xfr#11002, to-chk=0/11011)


In [6]:
!du -sh /content/LIPAD_CORROSION_DATASET

817M	/content/LIPAD_CORROSION_DATASET


In [7]:
from pathlib import Path

LOCAL_DATASET = Path("/content/LIPAD_CORROSION_DATASET")

print("Dataset exists:", LOCAL_DATASET.exists())

for split in ["train", "val", "test"]:
    images = LOCAL_DATASET / "images" / split
    labels = LOCAL_DATASET / "labels" / split

    print(
        f"{split}: "
        f"{len(list(images.glob('*'))) if images.exists() else 0} images, "
        f"{len(list(labels.glob('*'))) if labels.exists() else 0} labels"
    )

Dataset exists: True
train: 5163 images, 5163 labels
val: 223 images, 223 labels
test: 115 images, 115 labels


In [9]:
%cd /content

!rm -rf LIPAD_YOLO_TRAINING

!git clone https://github.com/sarieljandaniel-svg/Corrosion.git

/content
Cloning into 'Corrosion'...
remote: Enumerating objects: 16252, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 16252 (delta 39), reused 24 (delta 18), pack-reused 16186 (from 1)
Receiving objects: 100% (16252/16252), 1.58 GiB | 36.45 MiB/s, done.
Resolving deltas: 100% (1197/1197), done.
Updating files: 100% (16098/16098), done.


In [10]:
%cd /content/Corrosion/LIPAD_YOLO_TRAINING

!find . -maxdepth 2 -type f | sort | head -50

/content/Corrosion/LIPAD_YOLO_TRAINING
./corrosion_detection/colab_train.ipynb
./corrosion_detection/dataset.yaml
./corrosion_detection/evaluate_results.py
./corrosion_detection/gpu_check.py
./corrosion_detection/new_train_yolov8.py
./corrosion_detection/old_train_yolov8.py
./corrosion_detection/Python-3.12.13.tar.xz
./corrosion_detection/train_all.py
./corrosion_detection/train_yolov11.py
./corrosion_detection/train_yolov12.py
./corrosion_detection/train_yolov26.py
./corrosion_detection/yolo26n.pt
./corrosion_detection/yolov8m-seg.pt
./crack_detection/colab_train.ipynb
./crack_detection/dataset.yaml
./crack_detection/enhance_yolov8.py
./crack_detection/multivariant_training.py
./crack_detection/quantize_ensemble.py
./crack_detection/resume_training.py
./crack_detection/segregate_data.py
./crack_detection/subset_m.yaml
./crack_detection/subset_s.yaml
./crack_detection/subset_x.yaml
./crack_detection/train_all.py
./crack_detection/train_yolov11.py
./crack_detection/train_yolov12.py
./cr

In [11]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.6/120.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.3/915.3 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 7.3.0 which is incompatible.
jupyter-kernel-gateway 2.5.2 requires jupyter-client<8.0,>=5.2.0, but you have jupyter-client

In [12]:
import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.11.0+cu128
Ultralytics: 8.4.118
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [13]:
!cp shared/paths.py shared/paths.py.backup

In [14]:
from pathlib import Path

path_file = Path("shared/paths.py")
text = path_file.read_text()

old = '''def datasets_dir(task: str) -> Path:
    if is_colab() and task == "corrosion_detection":
        drive_dataset = Path(
            "/content/drive/MyDrive/"
            "LIPAD_TRAINING_VERSION2/datasets/corrosion/dataset"
        )

        if drive_dataset.exists():
            return drive_dataset

    return task_root(task) / "datasets"
'''

new = '''def datasets_dir(task: str) -> Path:
    if task == "corrosion_detection":
        local_dataset = os.environ.get("LIPAD_CORROSION_DATASET")

        if local_dataset:
            candidate = Path(local_dataset).expanduser().resolve()
            if candidate.exists():
                return candidate

    return task_root(task) / "datasets"
'''

if old not in text:
    raise RuntimeError("Expected datasets_dir() block was not found.")

path_file.write_text(text.replace(old, new))

print("paths.py updated successfully.")

paths.py updated successfully.


In [15]:
import os

os.environ["LIPAD_CORROSION_DATASET"] = "/content/LIPAD_CORROSION_DATASET"

print(os.environ["LIPAD_CORROSION_DATASET"])

/content/LIPAD_CORROSION_DATASET


In [16]:
import importlib
import shared.paths

importlib.reload(shared.paths)

from shared.paths import datasets_dir

print("datasets_dir():")
print(datasets_dir("corrosion_detection"))

datasets_dir():
/content/LIPAD_CORROSION_DATASET


In [17]:
from shared.preprocess_corrosion import write_corrosion_dataset_yaml

yaml_path = write_corrosion_dataset_yaml()

print("YAML:", yaml_path)
print()
print(yaml_path.read_text())

YAML: /content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml

path: /content/LIPAD_CORROSION_DATASET
train: images/train
val: images/val
nc: 3
names:
  0: fair
  1: poor
  2: severe



In [18]:
from pathlib import Path
import yaml

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

dataset_root = Path(data["path"])

print("Dataset root:", dataset_root)

for split in ["train", "val"]:
    image_dir = dataset_root / data[split]

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp",
            ".webp", ".tif", ".tiff"
        }
    ]

    print(f"{split}: {len(images)} images")

Dataset root: /content/LIPAD_CORROSION_DATASET
train: 5163 images
val: 223 images


In [19]:
from pathlib import Path

CORROSION_RUNS = Path(
    "/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs"
)

CORROSION_RUNS.mkdir(parents=True, exist_ok=True)

print("Training results will be saved to:")
print(CORROSION_RUNS)

Training results will be saved to:
/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs


In [20]:
import importlib

import shared.paths
import shared.preprocess_corrosion
import shared.trainer

importlib.reload(shared.paths)
importlib.reload(shared.preprocess_corrosion)
importlib.reload(shared.trainer)

from shared.trainer import train_corrosion

print("Trainer loaded.")

Trainer loaded.


In [21]:
from shared.model_registry import get_model_spec

spec = get_model_spec("corrosion", "yolov8")

print("Weights:", spec.weights)
print("Description:", spec.description)

Weights: yolov8m-seg.pt
Description: YOLOv8 medium segmentation (corrosion)


In [22]:
import torch
from shared.paths import datasets_dir

print("========== FINAL CHECK ==========")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())
print("Dataset:", datasets_dir("corrosion_detection"))
print("Runs:", CORROSION_RUNS)
print("Model:", spec.weights)
print("================================")

========== FINAL CHECK ==========
GPU: Tesla T4
CUDA: True
Dataset: /content/LIPAD_CORROSION_DATASET
Runs: /content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs
Model: yolov8m-seg.pt


In [ ]:
best_v8 = train_corrosion(
    "yolov8",
    batch=4,
    device=0,
    workers=4,
    project_name=str(CORROSION_RUNS),
    run_name="yolov8m_baseline",
    resume=False,
)

print("Best weights:", best_v8)

Base training
  epochs=100, lr0=0.01, optimizer=SGD
Preprocessing
  auto_orient=True
  resize=640x640 (stretch)
  clahe=True (adaptive equalization)
  class_remap=4 ids, drop=[2, 5, 6]
  offline_augment_copies=3
Augmentations (training + optional offline preprocess)
  flips: horizontal + vertical
  rotation: ±15.0°
  exposure: ±25%
  blur: up to 2.5px
  noise: up to 10% of pixels
  classes: {0: 'fair', 1: 'poor', 2: 'severe'}
[LiPAD] Task      : corrosion
[LiPAD] Model     : yolov8m-seg.pt (YOLOv8 medium segmentation (corrosion))
[LiPAD] Data      : /content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml
[LiPAD] Device    : 0
[LiPAD] Colab     : True


/usr/local/lib/python3.12/dist-packages/albumentations/augmentations/blur/functional.py:231: UserWarning: blur_limit: Invalid kernel size range (1, 3). Values less than 3 are not allowed. Range automatically adjusted to (3, 3).
  result = _ensure_min_value(result, min_value, info.field_name)


Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[Blur(p=0.5, blur_limit=(3, 3)), GaussNoise(p=0.5, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.0, 0.1))], auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.25, imgsz=640, iou=0.7, keras=False, k

In [23]:
from pathlib import Path

DRIVE_DATASET = Path(
    "/content/drive/MyDrive/LIPAD_TRAINING_VERSION2/"
    "datasets/corrosion/dataset"
)

LOCAL_DATASET = Path("/content/LIPAD_CORROSION_DATASET")

print("Drive dataset:", DRIVE_DATASET)
print("Local destination:", LOCAL_DATASET)
print("Drive dataset exists:", DRIVE_DATASET.exists())

Drive dataset: /content/drive/MyDrive/LIPAD_TRAINING_VERSION2/datasets/corrosion/dataset
Local destination: /content/LIPAD_CORROSION_DATASET
Drive dataset exists: True


In [24]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LIPAD_TRAINING_VERSION2")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Google Drive training directory:", DRIVE_ROOT)
print("Exists:", DRIVE_ROOT.exists())

Google Drive training directory: /content/drive/MyDrive/LIPAD_TRAINING_VERSION2
Exists: True


In [27]:
!find /content -maxdepth 4 -type f -name "corrosion_config.py" 2>/dev/null

/content/Corrosion/LIPAD_YOLO_TRAINING/shared/corrosion_config.py


In [28]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/Corrosion/LIPAD_YOLO_TRAINING")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Shared exists:", (PROJECT_ROOT / "shared").exists())
print(
    "Config exists:",
    (PROJECT_ROOT / "shared" / "corrosion_config.py").exists()
)

Project root: /content/Corrosion/LIPAD_YOLO_TRAINING
Shared exists: True
Config exists: True


In [45]:
!find /content/Corrosion/LIPAD_YOLO_TRAINING -maxdepth 2 -type f | sort

/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/colab_train.ipynb
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/evaluate_results.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/gpu_check.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/new_train_yolov8.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/old_train_yolov8.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/Python-3.12.13.tar.xz
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/train_all.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/train_yolov11.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/train_yolov12.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/train_yolov26.py
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/yolo26n.pt
/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/yolov8m-seg.pt
/content/Corrosio

In [46]:
!sed -n '1,260p' /content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/new_train_yolov8.py

"""Train YOLOv8 corrosion segmentation (Ameli et al. 2024 methodology)."""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.trainer import train

if __name__ == "__main__":
    train(
        task="corrosion",
        family="yolov8",

        # Paper settings
        epochs=250,
        imgsz=640,

        # Hardware limitation
        batch=2,

        # Optimization
        optimizer="AdamW",
        lr0=0.0001,
        lrf=0.01,
        warmup_epochs=5,
        warmup_momentum=0.8,
        weight_decay=0.0005,

        # Simulate paper batch behaviour
        nbs=8,

        # Reproducibility
        seed=0,
        deterministic=True,
        pretrained=True,

        # Device
        device=0,
        workers=2,

        # Augmentation
        mosaic=1.0,
        close_mosaic=10,
        fliplr=0.5,
        flipud=0.0,
        scale=0.5,
        translate=0.1,
        hsv_

In [47]:
!sed -n '1,260p' /content/Corrosion/LIPAD_YOLO_TRAINING/shared/trainer.py

"""Unified Ultralytics training entrypoint."""

from __future__ import annotations

import argparse
import os
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

from shared.model_registry import ModelFamily, TaskType, all_families, get_model_spec
from shared.paths import dataset_yaml_path, ensure_dataset_layout, is_colab, runs_dir, task_root

try:
    from shared.corrosion_config import CorrosionTrainConfig, config_summary, corrosion_train_kwargs
    from shared.preprocess_corrosion import write_corrosion_dataset_yaml
except ImportError:
    CorrosionTrainConfig = None  # type: ignore[misc, assignment]
    config_summary = None  # type: ignore[assignment]
    corrosion_train_kwargs = None  # type: ignore[assignment]
    write_corrosion_dataset_yaml = None  # type: ignore[assignment]


def write_dataset_yaml(task: TaskType, class_name: str) -> Path:
    root = task_root(f"{task}_detection")
    ensure_dataset_layout(f"{task}_detection")
    yaml_path = data

In [48]:
!sed -n '1,260p' /content/Corrosion/LIPAD_YOLO_TRAINING/shared/corrosion_config.py

"""Corrosion detection training and preprocessing configuration."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

# Matches Roboflow export: 2 classes remapped, 3 dropped → 3 severity classes kept.
CORROSION_CLASS_NAMES: dict[int, str] = {
    0: "fair",
    1: "poor",
    2: "severe",
}

# Map raw label class ids (from an unprocessed export) to remapped ids.
# Drop any class id listed in CORROSION_DROP_CLASS_IDS.
CORROSION_CLASS_REMAP: dict[int, int] = {
    0: 0,  # fair
    1: 1,  # poor
    3: 2,  # severe (example: original id 3 remapped)
    4: 2,  # severe (example: original id 4 remapped)
}
CORROSION_DROP_CLASS_IDS: frozenset[int] = frozenset({2, 5, 6})


@dataclass(frozen=True)
class CorrosionPreprocessConfig:
    image_size: int = 640
    apply_auto_orient: bool = True
    apply_clahe: bool = True
    clahe_clip_limit: float = 2.0
    clahe_tile_grid_size: tuple[int, int] = (8, 8)
    resize_mode: str = "stretch"  # Roboflow: S

In [49]:
from pathlib import Path

LAST_PT = Path(
    "/content/drive/MyDrive/"
    "LIPAD_TRAINING/corrosion/runs/"
    "yolov8m_baseline/weights/last.pt"
)

print("last.pt exists:", LAST_PT.exists())

if LAST_PT.exists():
    print("Size:", round(LAST_PT.stat().st_size / 1024**2, 2), "MB")

last.pt exists: True
Size: 104.38 MB


In [50]:
from pathlib import Path

# Search the mounted Drive for the checkpoint
matches = list(Path("/content/drive").rglob("last.pt"))

print("Found:", len(matches))

for p in matches:
    print(p)
    print(f"  Size: {p.stat().st_size / (1024**2):.2f} MB")

Found: 1
/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt
  Size: 104.38 MB


In [52]:
from ultralytics import YOLO

LAST_PT = "/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt"

model = YOLO(LAST_PT)

print(model.ckpt_path if hasattr(model, "ckpt_path") else "Checkpoint loaded")

/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt


In [53]:
from ultralytics import YOLO

LAST_PT = "/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt"

model = YOLO(LAST_PT)

print("Checkpoint loaded successfully.")
print("Model:", model.model.__class__.__name__)

if hasattr(model, "ckpt") and model.ckpt:
    print("Checkpoint keys:", model.ckpt.keys())
    print("Epoch:", model.ckpt.get("epoch"))

Checkpoint loaded successfully.
Model: SegmentationModel
Checkpoint keys: dict_keys(['epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'scaler', 'train_args', 'train_metrics', 'train_results', 'date', 'version', 'git', 'license', 'docs'])
Epoch: 76


In [54]:
from ultralytics import YOLO

LAST_PT = "/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt"

model = YOLO(LAST_PT)

ckpt = model.ckpt

print("Checkpoint loaded:", LAST_PT)
print("Epoch stored:", ckpt.get("epoch"))
print("Best fitness:", ckpt.get("best_fitness"))
print("Training args:")
print(ckpt.get("train_args"))

Checkpoint loaded: /content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt
Epoch stored: 76
Best fitness: 0.7233
Training args:
{'task': 'segment', 'mode': 'train', 'model': '/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt', 'data': '/content/Corrosion/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml', 'epochs': 100, 'time': None, 'patience': 100, 'batch': 4, 'imgsz': 640, 'save': True, 'save_period': -1, 'cache': False, 'device': '0', 'workers': 4, 'project': '/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs', 'name': 'yolov8m_baseline', 'exist_ok': False, 'pretrained': True, 'cls_remap': True, 'optimizer': 'SGD', 'verbose': True, 'seed': 0, 'deterministic': True, 'single_cls': False, 'rect': False, 'cos_lr': False, 'close_mosaic': 0, 'resume': '/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt', 'amp': True, 'fraction': 1.0,

In [55]:
from pathlib import Path

LAST_PT = Path(
    "/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/"
    "LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt"
)

print("Checkpoint exists:", LAST_PT.exists())
print("Checkpoint size:", round(LAST_PT.stat().st_size / (1024**2), 2), "MB")

Checkpoint exists: True
Checkpoint size: 104.38 MB


In [ ]:
from ultralytics import YOLO

LAST_PT = "/content/drive/.shortcut-targets-by-id/1MJAzXkfJTMnAzUKxO8QA3itJz2CsFaMV/LIPAD_TRAINING/corrosion/runs/yolov8m_baseline/weights/last.pt"

print("Loading checkpoint...")
model = YOLO(LAST_PT)

print("Resuming YOLOv8m-seg training...")
model.train(
    resume=LAST_PT
)

Loading checkpoint...
Resuming YOLOv8m-seg training...
WARNING ⚠️ Custom Albumentations transforms were used in the original training run but are not being restored. To preserve custom augmentations when resuming, you need to pass the 'augmentations' parameter again to get expected results. Example: 
model.train(resume=True, augmentations=[Blur(p=0.5, blur_limit=(3, 3)), GaussNoise(p=0.5, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.0, 0.1))])
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[Blur(p=0.5, blur_limit=(3, 3)), GaussNoise(p=0.5, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.0, 0.1))], auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, cop

In [31]:
!find /content -type f -name "requirements.txt" 2>/dev/null

/content/Corrosion/LIPAD_YOLO_TRAINING/requirements.txt
/content/Corrosion/PROJECT_LIPAD/requirements.txt
/content/Corrosion/PROJECT_LIPAD/Project_LIPAD_AI/yolov5/requirements.txt


In [32]:
!find /content -maxdepth 4 -type d -name "Corrosion" 2>/dev/null

/content/Corrosion


In [33]:
!ls -lah /content

total 28K
drwxr-xr-x 1 root root 4.0K Aug 12 07:41 .
drwxr-xr-x 1 root root 4.0K Aug 12 05:51 ..
drwxr-xr-x 4 root root 4.0K Aug 10 13:31 .config
drwxr-xr-x 5 root root 4.0K Aug 12 07:42 Corrosion
drwx------ 5 root root 4.0K Aug 12 06:01 drive
drwx------ 4 root root 4.0K Aug 11 06:15 LIPAD_CORROSION_DATASET
drwxr-xr-x 1 root root 4.0K Aug 10 13:31 sample_data


In [34]:
%pip install -r /content/Corrosion/LIPAD_YOLO_TRAINING/requirements.txt

In [35]:
import inspect
from shared.trainer import train_corrosion

print(inspect.signature(train_corrosion))

(family: 'ModelFamily', *, config: "'CorrosionTrainConfig | None'" = None, batch: 'int | None' = None, device: 'str | int | None' = None, workers: 'int | None' = None, project_name: 'str | None' = None, run_name: 'str | None' = None, resume: 'bool' = False) -> 'Path'


In [36]:
import inspect
from shared.trainer import train_corrosion

print(inspect.getsource(train_corrosion))

def train_corrosion(
    family: ModelFamily,
    *,
    config: "CorrosionTrainConfig | None" = None,
    batch: int | None = None,
    device: str | int | None = None,
    workers: int | None = None,
    project_name: str | None = None,
    run_name: str | None = None,
    resume: bool = False,
) -> Path:
    """Train corrosion segmentation with Roboflow-aligned hyperparameters."""
    if corrosion_train_kwargs is None or write_corrosion_dataset_yaml is None:
        raise ImportError("shared.corrosion_config is required for train_corrosion()")

    task_dir = "corrosion_detection"
    data_yaml = write_corrosion_dataset_yaml()
    spec = get_model_spec("corrosion", family)

    if device is None:
        device = 0 if torch.cuda.is_available() else "cpu"
    if workers is None:
        workers = 2 if os.name == "nt" else 4
        if device == "cpu":
            workers = 0

    project = project_name or str(runs_dir(task_dir) / family)
    name = run_name or f"corrosion_{family}_se

In [37]:
import inspect
from shared.trainer import train_corrosion

source = inspect.getsource(train_corrosion)

for i, line in enumerate(source.splitlines(), 1):
    if "project" in line.lower() or "run_name" in line.lower() or "model.train" in line.lower():
        print(f"{i:3}: {line}")

  8:     project_name: str | None = None,
  9:     run_name: str | None = None,
 27:     project = project_name or str(runs_dir(task_dir) / family)
 28:     name = run_name or f"corrosion_{family}_seg"
 42:         project=project,
 48:     model.train(data=str(data_yaml), **train_kwargs)
 50:     best = Path(project) / name / "weights" / "best.pt"


In [38]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LIPAD_TRAINING")

CORROSION_RUNS = DRIVE_ROOT / "corrosion" / "runs"
CORROSION_RUNS.mkdir(parents=True, exist_ok=True)

print("Saving corrosion training to:")
print(CORROSION_RUNS)

Saving corrosion training to:
/content/drive/MyDrive/LIPAD_TRAINING/corrosion/runs


In [ ]:
CORROSION_DRIVE = DRIVE_ROOT / "corrosion"
CRACK_DRIVE = DRIVE_ROOT / "crack"

for directory in [
    CORROSION_DRIVE / "weights",
    CORROSION_DRIVE / "runs",
    CORROSION_DRIVE / "results",
    CRACK_DRIVE / "weights",
    CRACK_DRIVE / "runs",
    CRACK_DRIVE / "results",
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Training directories created.")

Training directories created.


In [ ]:
!git clone https://github.com/sarieljandaniel-svg/Corrosion.git /content/LIPAD_YOLO_TRAINING

Cloning into '/content/LIPAD_YOLO_TRAINING'...
remote: Enumerating objects: 16223, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 16223 (delta 16), reused 21 (delta 14), pack-reused 16186 (from 1)
Receiving objects: 100% (16223/16223), 1.58 GiB | 18.88 MiB/s, done.
Resolving deltas: 100% (1174/1174), done.
Updating files: 100% (16098/16098), done.


In [ ]:
%cd /content/LIPAD_YOLO_TRAINING
!git status

[Errno 2] No such file or directory: '/content/LIPAD_YOLO_TRAINING'
/content
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!git clone https://github.com/sarieljandaniel-svg/Corrosion.git /content/LIPAD_YOLO_TRAINING

fatal: not a git repository (or any of the parent directories): .git


In [41]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/Corrosion/LIPAD_YOLO_TRAINING")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())
print("Shared exists:", (PROJECT_ROOT / "shared").exists())
print(
    "Config exists:",
    (PROJECT_ROOT / "shared" / "corrosion_config.py").exists()
)

Project root: /content/Corrosion/LIPAD_YOLO_TRAINING
Exists: True
Shared exists: True
Config exists: True


In [44]:
!ls -la /content/Corrosion/LIPAD_YOLO_TRAINING/

total 48
drwxr-xr-x 5 root root 4096 Aug 12 07:42 .
drwxr-xr-x 5 root root 4096 Aug 12 07:42 ..
drwxr-xr-x 5 root root 4096 Aug 12 07:42 corrosion_detection
drwxr-xr-x 4 root root 4096 Aug 12 07:42 crack_detection
-rw-r--r-- 1 root root   64 Aug 12 07:42 .gitattributes
-rw-r--r-- 1 root root  353 Aug 12 07:42 .gitignore
-rw-r--r-- 1 root root  246 Aug 12 07:42 LIPAD_YOLO_TRAINING.code-workspace
-rw-r--r-- 1 root root 1462 Aug 12 07:42 one_epoch.py
-rw-r--r-- 1 root root 2037 Aug 12 07:42 README.md
-rw-r--r-- 1 root root  238 Aug 12 07:42 requirements.txt
-rw-r--r-- 1 root root  449 Aug 12 07:42 setup.ps1
drwxr-xr-x 3 root root 4096 Aug 12 07:46 shared


In [ ]:
# @title 2. Review training configuration

from shared.corrosion_config import (
    config_summary,
    CorrosionPreprocessConfig,
    CorrosionTrainConfig,
)

print(config_summary())

print("\nPreprocess config:", CorrosionPreprocessConfig())
print("Train config     :", CorrosionTrainConfig())

Base training
  epochs=100, lr0=0.01, optimizer=SGD
Preprocessing
  auto_orient=True
  resize=640x640 (stretch)
  clahe=True (adaptive equalization)
  class_remap=4 ids, drop=[2, 5, 6]
  offline_augment_copies=3
Augmentations (training + optional offline preprocess)
  flips: horizontal + vertical
  rotation: ±15.0°
  exposure: ±25%
  blur: up to 2.5px
  noise: up to 10% of pixels
  classes: {0: 'fair', 1: 'poor', 2: 'severe'}

Preprocess config: CorrosionPreprocessConfig(image_size=640, apply_auto_orient=True, apply_clahe=True, clahe_clip_limit=2.0, clahe_tile_grid_size=(8, 8), resize_mode='stretch', offline_augment_copies=3, horizontal_flip=True, vertical_flip=True, rotation_degrees=15.0, exposure_fraction=0.25, blur_max_pixels=2.5, noise_max_pixel_fraction=0.1)
Train config     : CorrosionTrainConfig(epochs=100, lr0=0.01, optimizer='SGD', imgsz=640, batch=16, horizontal_flip_prob=0.5, vertical_flip_prob=0.5, rotation_degrees=15.0, exposure_fraction=0.25, blur_max_pixels=2.5, noise_ma

In [ ]:
!find /content/LIPAD_YOLO_TRAINING -maxdepth 5 -type f -name "corrosion_config.py"

/content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/shared/corrosion_config.py


In [ ]:
!find /content/LIPAD_YOLO_TRAINING -maxdepth 4 -type d -name "shared"

/content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/shared


In [ ]:
!ls -la /content/LIPAD_YOLO_TRAINING

total 456
drwxr-xr-x  5 root root   4096 Aug 11 05:47 .
drwxr-xr-x  1 root root   4096 Aug 11 05:46 ..
drwxr-xr-x  8 root root   4096 Aug 11 05:47 .git
drwxr-xr-x  5 root root   4096 Aug 11 05:47 LIPAD_YOLO_TRAINING
drwxr-xr-x 10 root root   4096 Aug 11 05:47 PROJECT_LIPAD
-rw-r--r--  1 root root 443933 Aug 11 05:47 Project_LiPAD.ipynb


In [ ]:
# @title 3. Prepare dataset folders
from shared.paths import ensure_dataset_layout, datasets_dir, task_root

ensure_dataset_layout('corrosion_detection')
raw_root = task_root('corrosion_detection') / 'datasets_raw'
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (raw_root / sub).mkdir(parents=True, exist_ok=True)

print('Processed dataset :', datasets_dir('corrosion_detection'))
print('Optional raw input  :', raw_root)
print('Expected YOLO layout:')
print('  datasets_raw/images/train  +  datasets_raw/labels/train')
print('  datasets_raw/images/val    +  datasets_raw/labels/val')
print('Classes: fair (0), poor (1), severe (2)')

Processed dataset : /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/datasets
Optional raw input  : /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/datasets_raw
Expected YOLO layout:
  datasets_raw/images/train  +  datasets_raw/labels/train
  datasets_raw/images/val    +  datasets_raw/labels/val
Classes: fair (0), poor (1), severe (2)


In [ ]:
# @title 4. Preprocess (auto-orient, stretch 640, CLAHE, class remap, 3x train copies)
from shared.preprocess_corrosion import preprocess_corrosion_dataset
from shared.paths import datasets_dir, task_root

RAW_DIR = task_root('corrosion_detection') / 'datasets_raw'
USE_RAW = any((RAW_DIR / 'images' / 'train').glob('*'))

yaml_path = preprocess_corrosion_dataset(
    raw_dir=RAW_DIR if USE_RAW else None,
    clear_existing=True,
)
print('Wrote dataset yaml:', yaml_path)
if USE_RAW:
    print('Raw folder used:', RAW_DIR)
else:
    print('In-place reprocess of:', datasets_dir('corrosion_detection'))

Wrote dataset yaml: /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/dataset.yaml
In-place reprocess of: /content/LIPAD_YOLO_TRAINING/LIPAD_YOLO_TRAINING/corrosion_detection/datasets


In [ ]:
from shared.model_registry import get_model_spec

spec = get_model_spec("corrosion", "yolov8")

print(spec)

ModelSpec(family='yolov8', weights='yolov8n-seg.pt', description='YOLOv8 nano segmentation (corrosion)')


In [ ]:
%cd /content/LIPAD_YOLO_TRAINING

/content/LIPAD_YOLO_TRAINING


In [ ]:
%cd /content/LIPAD_YOLO_TRAINING
!git pull

/content/LIPAD_YOLO_TRAINING
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.97 KiB | 673.00 KiB/s, done.
From https://github.com/sarieljandaniel-svg/Corrosion
   81c04e6..9d95863  main       -> origin/main
Updating 81c04e6..9d95863
Fast-forward
 .../corrosion_detection/colab_train.ipynb          | 291 +++++++++++----------
 LIPAD_YOLO_TRAINING/shared/model_registry.py       |   2 +-
 2 files changed, 158 insertions(+), 135 deletions(-)


In [ ]:
import importlib
import shared.model_registry

importlib.reload(shared.model_registry)

spec = shared.model_registry.get_model_spec("corrosion", "yolov8")
print(vars(spec))

{'family': 'yolov8', 'weights': 'yolov8m-seg.pt', 'description': 'YOLOv8 medium segmentation (corrosion)'}


In [ ]:
!nvidia-smi

Tue Aug 11 06:56:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from shared.trainer import train_corrosion

best_v8 = train_corrosion(
    "yolov8",
    batch=4,
    project_name=str(CORROSION_RUNS),
    run_name="yolov8m_baseline",
)

print("Done:", best_v8)

In [ ]:
# @title 6. Train YOLOv11 (configured)
from shared.trainer import train_corrosion

best_v11 = train_corrosion('yolov11', batch=16 if IN_COLAB else 8)
print('Done:', best_v11)

In [ ]:
# @title 7. Train YOLOv12 (configured)
from shared.trainer import train_corrosion

best_v12 = train_corrosion('yolov12', batch=16 if IN_COLAB else 8)
print('Done:', best_v12)

## Step-by-step training guide

### A. Prepare your data

**Option 1 — Roboflow export (recommended)**

1. Export from Roboflow as **YOLOv8 Segmentation**.
2. Match these Roboflow settings if possible:
   - Auto-Orient: on
   - Resize: Stretch 640×640
   - Auto-Adjust Contrast: Adaptive Equalization
   - Modify Classes: 2 remapped, 3 dropped
   - Augmentations: 3 outputs, flips, ±15° rotation, ±25% exposure, blur ≤2.5 px, noise ≤10%
3. Unzip into `corrosion_detection/datasets_raw/` with this layout:

```
datasets_raw/
  images/train/
  images/val/
  labels/train/
  labels/val/
```

**Option 2 — Already in YOLO layout**

Place files directly under `corrosion_detection/datasets/images/{train,val}` and `labels/{train,val}`. Cell 4 will reprocess them in place.

### B. Adjust class mapping (if needed)

Open `shared/corrosion_config.py` and edit:

- `CORROSION_CLASS_REMAP` — map old class ids to fair/poor/severe
- `CORROSION_DROP_CLASS_IDS` — ids to ignore

Default output classes: **fair (0), poor (1), severe (2)**.

### C. Run the notebook

| Cell | Action |
|---|---|
| 1 | Install deps + set repo path |
| 2 | Print active configuration |
| 3 | Create folder structure |
| 4 | Preprocess images (orient, stretch, CLAHE, remap, 3 train copies) |
| 5–7 | Train YOLOv8 / v11 / v12 (pick one or run all) |

### D. After training

- Weights: `corrosion_detection/runs/<model>/corrosion_<model>_seg/weights/best.pt`
- Metrics/plots: same run folder
- Download `best.pt` from Colab: Files panel → right-click → Download

### E. Notes

- **3 outputs per example:** offline copies use photometric aug (exposure/blur/noise). Flips and rotation are applied during YOLO training with correct mask transforms.
- **Batch size:** lower to `8` or `4` if Colab runs out of GPU memory.
- **Resume training:** call `train_corrosion('yolov8', resume=True)`.